# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | 2026-05-04 & 2026-05-11 |
| Datum (Phase 2) | _YYYY-MM-DD_ |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | 44 |
| Genutzte Suchanfragen (Phase 1) | Data Analyst, Data Engineer, Business Intelligence, Fachinformatiker Daten- und Prozessanalyse, Data Scientist -> jeweils in Bremen |
| Pair-Partner:in (Phase 2) | _ |
| 12 gemeinsame Anzeigen-IDs | _ |

## Phase 1 — Korpus inspizieren + Bias-Notiz

**Beschaffen der Daten über API Request und generieren des Korpus**

In [4]:
import requests
import json
import time
import base64
from pathlib import Path

API_KEY = "jobboerse-jobsuche"
BASE_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4"

HEADERS = {
    "X-API-Key": API_KEY
}

suchanfragen = [
    {"was": "Data Analyst", "wo": "Bremen"},
    {"was": "Data Engineer", "wo": "Bremen"},
    {"was": "Business Intelligence", "wo": "Bremen"},
    {"was": "Fachinformatiker Daten- und Prozessanalyse", "wo": "Bremen"},
    {"was": "Data Scientist", "wo": "Bremen"},
]

anzeigen = {}

for suche in suchanfragen:
    print("Suche:", suche)

    response = requests.get(
        f"{BASE_URL}/jobs",
        headers=HEADERS,
        params={
            "was": suche["was"],
            "wo": suche["wo"],
            "size": 20,
        }
    )

    print("Status:", response.status_code)

    daten = response.json() # dictonary aus Antwort erstellen
    treffer = daten.get("stellenangebote", []) # Liste mit Stellenangeboten ziehen
    
# Detailsuche für gefundene Treffer anhand Refnr um an stellenangebotsBeschreibung zu kommen
    for treffer_item in treffer:
        refnr = treffer_item.get("refnr")

        if not refnr:
            continue

        if refnr in anzeigen: # falls Anzeige bereits mit anderem Suchbegriff gefunden wurde und doppelt auftaucht
            continue

        refnr_encoded = base64.b64encode(refnr.encode("utf-8")).decode("utf-8") # damit das Anhängen an die URL keine Fehler durch Sonderzeichen wirft

        detail_response = requests.get(
            f"{BASE_URL}/jobdetails/{refnr_encoded}",
            headers=HEADERS
        )

        if detail_response.status_code != 200:
            print("Detail fehlgeschlagen:", refnr, detail_response.status_code)
            continue

        detail = detail_response.json()

        anzeige = {
            "refnr": refnr,
            "titel": detail.get("stellenangebotsTitel"),
            "firma": detail.get("firma"),
            "text": detail.get("stellenangebotsBeschreibung"),
            "ort": detail.get("stellenlokationen", [{}])[0].get("adresse", {}).get("ort"),
            "beruf": detail.get("hauptberuf"),
            "veroeffentlichung": detail.get("datumErsteVeroeffentlichung"),
            "homeoffice_api": detail.get("homeofficemoeglich"),
            "gehalt_api": detail.get("verguetungsangabe"),
            "vertragsdauer_api": detail.get("vertragsdauer"),
            "raw": detail # Originalantwort der API
        }

        if anzeige["text"]: #anzeige nur im dic speichern, wenn tatsächlich stellenbeschreibung vorhanden
            anzeigen[refnr] = anzeige

        time.sleep(0.5) # Pause zwischen Abfragen

print("Gesammelte Anzeigen:", len(anzeigen))

Suche: {'was': 'Data Analyst', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Data Engineer', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Business Intelligence', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Fachinformatiker Daten- und Prozessanalyse', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Data Scientist', 'wo': 'Bremen'}
Status: 200
Gesammelte Anzeigen: 44


In [5]:
output_path = Path("../daten/eigener_korpus.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f: #w -> überschreiben
    for anzeige in anzeigen.values():
        f.write(json.dumps(anzeige, ensure_ascii=False) + "\n") # ensure_ascii=False -> Umlaute erlauben; + "\n" danach neue Zeile

print("Gespeichert:", output_path)
print("Anzahl gespeicherter Anzeigen:", len(anzeigen))

Gespeichert: ../daten/eigener_korpus.jsonl
Anzahl gespeicherter Anzeigen: 44


In [7]:
with open("../daten/eigener_korpus.jsonl", "r", encoding="utf-8") as f:
    zeilen = f.readlines()

print("Zeilen in Datei:", len(zeilen))

erste_anzeige = json.loads(zeilen[0])
erste_anzeige.keys()

Zeilen in Datei: 44


dict_keys(['refnr', 'titel', 'firma', 'text', 'ort', 'beruf', 'veroeffentlichung', 'homeoffice_api', 'gehalt_api', 'vertragsdauer_api', 'raw'])

**Korpus Exploration**

In [9]:
import pandas as pd

korpus = pd.read_json("../daten/eigener_korpus.jsonl", lines=True)
korpus.head()

,refnr,titel,firma,text,ort,beruf,veroeffentlichung,homeoffice_api,gehalt_api,vertragsdauer_api,raw
0,10001-1002993453-S,Data Scientist/Analyst (m/w/d),SThree Temp Experts GmbH,## **Deine Aufgaben**\n\n- Du analysierst und ...,Bremen,Data-Analyst/in,2026-04-28,NaN,JAHRESGEHALT,BEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
1,10001-1002928664-S,Data Analyst (m/w/d),Wolters Rundreisen GmbH Sachbearbeiter/in,**STARTE MIT UNS DEINE NEUE REISE – WIR FREUEN...,Stuhr,Business-Analyst/in,2026-04-16,1.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
2,10001-1003016654-S,Data Analyst (m/w/d) Schwerpunkt BI,Orange Engineering GmbH & Co. KG,Im Auftrag unseres namhaften Kunden aus dem Ra...,Bremen,Data-Analyst/in,2026-05-04,NaN,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
3,10001-1002870609-S,Data Analyst / Data Engineer (m/w/d),Team Business IT GmbH,"## Aufgaben, die deine Neugier wecken:\n\nDu h...",Rostock,Data Engineer,2026-04-02,1.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
4,10001-1003011936-S,Data Analyst (m/w/d) für AIRBUS,SimpleXX GmbH,**Data Analyst (m/w/d) für AIRBUS**\n\n\_\_\_\...,Bremen,Bachelor Professional - IT (Datenanalyse),2026-05-02,NaN,KEINE_ANGABEN,KEINE_ANGABE,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."


Erste Auffälligkeiten:
- bei Ort taucht Stuhr und Rostock auf obwohl in Bremen gesucht werden sollte
- Index 4 hat im Titel ...für AIRBUS stehen und bei Firma SimpleXXGmbH (Könnte sich um ein Headhunter Unternehmen handeln) -> möglicher Kandidat für EdgeCase, da mehrere Interpretationen möglich
- einige Texte scheinen mit Markdown-Sonderzeichen durchzogen zu sein

In [10]:
korpus["text"].str.len().describe()

count      44.000000
mean     2848.772727
std      1093.728695
min       899.000000
25%      2217.750000
50%      2698.000000
75%      3258.750000
max      5678.000000
Name: text, dtype: float64

Hohe Standardabweichung bei Textlänge der Stellenbeschreibung
-> Texte unterscheiden sich sich stark in Länge und bewegen sich zwischen 899 und 5678 Zeichenlänge

In [17]:
cols_toCheck = ['titel', 'firma', 'ort', 'beruf', 'veroeffentlichung', 'homeoffice_api', 'gehalt_api', 'vertragsdauer_api']
for col in cols_toCheck:
    print(f"\nSpalte: {col}")
    print(korpus[col].value_counts(dropna=False))



Spalte: titel
titel
Electrical Engineer (m/w/d)                                                                           2
Data Scientist/Analyst (m/w/d)                                                                        1
Controller (m/w/d)                                                                                    1
Requirements Engineer (m/w/d)                                                                         1
Senior QA Engineer (m/w/d) – Testautomatisierung & OT-Softwarequalität - ab sofort - VZ               1
Technical Sales Engineer Mitte (Mittel- Südhessen bis Saarland) (w/m/d) - ab sofort - VZ              1
Software Engineer DevOps (m/w/d)                                                                      1
Geschäftsfeldsteuerer - Signal and Data Intelligence (m/w/d)                                          1
Duales Studium Bachelor of Science Angewandte Künstliche Intelligenz (w/d/m) - 2027                   1
Sachbearbeitung (m/w/d) Projektabrechnung /

Auffälligkeiten:
- __Jobtitel__ unterscheiden sich von der Syntax -> häufig Jobtitel + (m/w/d), aber auch genauere Spezifikationen, Jahreszahl, Vollzeit, Firma; Erfahrungslevel als Nennungen vorhanden; sogar 1x nur "**  Ausbildung 2026 **"; fast keine Mehrfachnennungen; einige Jobtitel wirken nicht so ganz passend z.B. Electrical Engineer und Sachbearbeitung (m/w/d) Projektabrechnung / Faktura 
- __Firma__ hat einige Mehrfachnennungen; auf ersten Blick keine Nennung von gleicher Firma mit unterschiedlicher Schreibweise; Rheinmetal 8x könnte Bias erzeugen
- __Ort__ hat noch mehr Orte neben Stuhr und Rostock, die nicht Bremen sind (insgesamt 7) -> Ursachenforschung (möglicherweise full remote?)
- __Veröffentlichung__ zeigt, dass die meisten Anzeigen aus den letzten Monaten stammen, vereinzelt auch 2025 vertreten
- __Homeoffice__ mit 0,1 oder NaN angegeben als float
- __Gehalt__ nur in 9 Fällen überhaupt Angaben
- __Vertragsdauer__ besser gepflegt, aber trotzdem noch 16 mal keine Angabe

-> drauf achten NaN oder KEINE_ANGABE
-> Senioritätslevel in diesen Daten nur vereinzelt im Jobtitel angegeben, sonst nirgendwo


In [ ]:
**Bias-Einschätzung**

## Phase 2 — κ-Tabelle + drei Edge Cases